# LeetCode 523 — Continuous Subarray Sum bằng Python

Link đề: https://leetcode.com/problems/continuous-subarray-sum/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Prefix Sum, Pigeonhole Principle

Skeleton LeetCode:

```python
class Solution:
    def checkSubarraySum(self, nums: List[int], k: int) -> bool:
        ...
```

Ghi chú: ví dụ 4, 5 trong mục 2 do notebook tự đặt để minh hoạ các trường hợp biên, đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **mảng con liên tiếp có độ dài ít nhất 2** mà **tổng chia hết cho `k`** (`0` cũng là bội của `k`), trả về `True/False`.
2. Nắm các công cụ Python: slicing `nums[i:j]`, toán tử `%`, `itertools.accumulate` để tính **tổng tiền tố (prefix sum)**, `dict`/`set` làm bảng băm, `enumerate`, và nguyên lý **ngăn kéo (pigeonhole)**.
3. Đi từ **vét cạn O(n³)** → **O(n²) với tổng chạy** → **tổng tiền tố theo modulo + dict lưu chỉ số đầu tiên (O(n))** → **set "trễ một nhịp"** → **DP tập các số dư** → **định lý ngăn kéo làm lối tắt**.
4. Hiểu mấu chốt: hai tổng tiền tố **cùng số dư khi chia `k`** thì đoạn nằm giữa chúng có tổng chia hết cho `k`; và **chỉ cần nhớ lần xuất hiện đầu tiên** của mỗi số dư.
5. Tránh các lỗi kinh điển: quên "mồi" `{0: -1}`, ghi đè chỉ số đầu tiên, quên điều kiện độ dài ≥ 2, mất số dư khi quên `% k`, và nhầm "bội của k" với "chia hết cho k khác 0".

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Trả về `True` nếu `nums` có một **mảng con tốt (good subarray)**, ngược lại trả về `False`.

Mảng con tốt là mảng con (một đoạn **liên tiếp** của `nums`) thoả:

- độ dài **ít nhất 2**, và
- tổng các phần tử là **bội của `k`**, tức tồn tại số nguyên `n` để tổng `= n * k`. Chú ý: `0` **luôn** là bội của `k`.

**Ví dụ 1**

- Input: `nums = [23,2,4,6,7]`, `k = 6`
- Output: `True`
- Giải thích: `[2, 4]` có độ dài 2 và tổng `6`.

**Ví dụ 2**

- Input: `nums = [23,2,6,4,7]`, `k = 6`
- Output: `True`
- Giải thích: cả mảng `[23,2,6,4,7]` dài 5 có tổng `42 = 7 * 6`.

**Ví dụ 3**

- Input: `nums = [23,2,6,4,7]`, `k = 13`
- Output: `False`

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `nums = [5]`, `k = 5`
- Output: `False`
- Giải thích: mảng con duy nhất là `[5]` có tổng chia hết cho 5 nhưng độ dài chỉ 1.

**Ví dụ 5 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `nums = [0,0]`, `k = 100`
- Output: `True`
- Giải thích: tổng `0` là bội của mọi `k`.

**Ràng buộc:**

- `1 <= nums.length <= 10^5`
- `0 <= nums[i] <= 10^9`
- `0 <= sum(nums[i]) <= 2^31 - 1`
- `1 <= k <= 2^31 - 1`

Chú ý: mọi phần tử **không âm** và `k >= 1`, nên ta không gặp `k = 0` hay số âm (nhưng mục 15 sẽ bàn khi các điều kiện này bị bỏ).

In [1]:
from typing import List
from itertools import accumulate, combinations
from collections import Counter
import random
import time

examples = [
    ([23, 2, 4, 6, 7], 6, True),
    ([23, 2, 6, 4, 7], 6, True),
    ([23, 2, 6, 4, 7], 13, False),
    ([5], 5, False),
    ([0, 0], 100, True),
]
for nums, k, expected in examples:
    print(f"nums={nums}, k={k} -> kỳ vọng {expected}")

nums=[23, 2, 4, 6, 7], k=6 -> kỳ vọng True
nums=[23, 2, 6, 4, 7], k=6 -> kỳ vọng True
nums=[23, 2, 6, 4, 7], k=13 -> kỳ vọng False
nums=[5], k=5 -> kỳ vọng False
nums=[0, 0], k=100 -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Mảng con liên tiếp và slicing

Mảng con (subarray) là đoạn **liền nhau** `nums[i], nums[i+1], ..., nums[j]`. Khác với "dãy con" (subsequence), ta **không được bỏ phần tử ở giữa**. Trong Python, `nums[i:j+1]` cắt đoạn từ chỉ số `i` đến `j` (cắt **không** gồm chỉ số cuối). Số phần tử của đoạn từ `i` đến `j` là `j - i + 1`; độ dài **ít nhất 2** nghĩa là `j - i >= 1`.

In [2]:
nums = [23, 2, 4, 6, 7]
print(nums[1:3])            # [2, 4]  -> chỉ số 1 và 2
print(sum(nums[1:3]))       # 6
i, j = 1, 2
print("độ dài =", j - i + 1)
print("số mảng con độ dài >= 2:", sum(1 for a in range(5) for b in range(a + 1, 5)))

[2, 4]
6
độ dài = 2
số mảng con độ dài >= 2: 10


### 3.2. "Bội của `k`" và toán tử `%`

`x` là bội của `k` khi `x % k == 0` (phép chia lấy dư bằng 0). Đặc biệt `0 % k == 0` với mọi `k >= 1`, nên tổng bằng `0` luôn tốt (ví dụ `[0, 0]`). Với số **không âm** và `k` dương, `x % k` luôn nằm trong `0 .. k-1`.

In [3]:
print(42 % 6, 13 % 6, 0 % 6, 7 % 13)
print([x % 6 for x in [23, 2, 4, 6, 7]])
print("0 có phải bội của 6 không?", 0 % 6 == 0)

0 1 0 7
[5, 2, 4, 0, 1]
0 có phải bội của 6 không? True


### 3.3. Tính chất của số dư (modulo)

Hai tính chất cần nhớ:

1. `(a + b) % k == ((a % k) + (b % k)) % k` — có thể **lấy dư giữa chừng**, kết quả không đổi. Nhờ đó ta chỉ cần giữ số nhỏ hơn `k`.
2. Nếu `a % k == b % k` thì `(a - b) % k == 0` — **hai số cùng số dư thì hiệu của chúng chia hết cho `k`**. Ví dụ `23` và `5` cùng dư `5` khi chia `6`, hiệu `18` chia hết cho `6`.

In [4]:
a, b, k = 23, 5, 6
print(a % k, b % k, (a - b) % k)
print((23 + 2) % 6, ((23 % 6) + (2 % 6)) % 6)

5 5 0
1 1


### 3.4. Tổng tiền tố (prefix sum)

Gọi `P[0] = 0` và `P[t] = nums[0] + ... + nums[t-1]` (tổng của `t` phần tử đầu). Khi đó tổng đoạn `nums[i..j]` bằng `P[j+1] - P[i]`: **một phép trừ thay cho cả vòng cộng**. `itertools.accumulate(nums, initial=0)` tạo đúng mảng `P` này.

In [5]:
nums = [23, 2, 4, 6, 7]
P = list(accumulate(nums, initial=0))
print("P =", P)
i, j = 1, 2
print("tổng nums[1..2] =", P[j + 1] - P[i], "| kiểm tra:", sum(nums[i:j + 1]))

P = [0, 23, 25, 29, 35, 42]
tổng nums[1..2] = 6 | kiểm tra: 6


### 3.5. `dict` làm bảng băm và `enumerate`

`dict` tra cứu `key in d` trung bình `O(1)`. Mẫu hay dùng: **chỉ lưu lần đầu** một khoá xuất hiện bằng `if key not in d: d[key] = value`. `enumerate(nums)` cho ta cặp `(chỉ số, giá trị)` mà khỏi dùng `range(len(...))`.

In [6]:
first = {}
for idx, ch in enumerate("abracadabra"):
    if ch not in first:          # chỉ ghi lần đầu
        first[ch] = idx
print(first)
print("'a' xuất hiện sớm nhất ở", first["a"], "| 'z' có trong dict?", "z" in first)

{'a': 0, 'b': 1, 'r': 2, 'c': 4, 'd': 6}
'a' xuất hiện sớm nhất ở 0 | 'z' có trong dict? False


### 3.6. `set` và nguyên lý ngăn kéo (pigeonhole)

`set` là tập hợp không trùng, `x in s` trung bình `O(1)`. **Nguyên lý ngăn kéo:** nhét `m` vật vào `r` ngăn mà `m > r` thì có ít nhất một ngăn chứa **từ 2 vật trở lên**; tổng quát nếu `m > 2r` thì có ngăn chứa **ít nhất 3 vật**. Ở bài này "ngăn" là số dư `0..k-1` (có `k` ngăn), "vật" là các tổng tiền tố. Ta sẽ dùng nó ở mục 6 và cách 5.

In [7]:
from collections import Counter
rems = [(x * 7) % 5 for x in range(11)]     # 11 vật, 5 ngăn
print(rems)
print(Counter(rems))
print("ngăn đông nhất có", max(Counter(rems).values()), "vật (>= ceil(11/5) = 3)")

[0, 2, 4, 1, 3, 0, 2, 4, 1, 3, 0]
Counter({0: 3, 2: 2, 4: 2, 1: 2, 3: 2})
ngăn đông nhất có 3 vật (>= ceil(11/5) = 3)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `nums = [23, 2, 4, 6, 7]`, `k = 6`. Cách nghĩ tự nhiên: **thử mọi đoạn có độ dài ≥ 2 rồi xem tổng có chia hết cho 6 không**.

| Đoạn | Tổng | `% 6` |
|---|---|---|
| `[23,2]` | 25 | 1 |
| `[2,4]` | 6 | **0** → tìm thấy |

Nhưng thay vì cộng lại từng đoạn, hãy ghi **tổng tiền tố** và **số dư** của nó:

| `t` | `P[t]` | `P[t] % 6` |
|---|---|---|
| 0 | 0 | 0 |
| 1 | 23 | 5 |
| 2 | 25 | 1 |
| 3 | 29 | 5 |
| 4 | 35 | 5 |
| 5 | 42 | 0 |

Nhìn cột cuối: số dư `5` xuất hiện ở `t = 1, 3, 4`. Chọn `t = 1` và `t = 3`: đoạn giữa là `nums[1..2] = [2, 4]` có tổng `29 - 23 = 6`. Số dư `0` xuất hiện ở `t = 0` và `t = 5`: cả mảng có tổng `42` chia hết cho 6. Trực giác: **tìm hai vị trí tổng tiền tố có cùng số dư**, và hai vị trí đó phải cách nhau ít nhất 2 để đoạn giữa có ≥ 2 phần tử.

In [8]:
nums, k = [23, 2, 4, 6, 7], 6
P = list(accumulate(nums, initial=0))
print("t      :", list(range(len(P))))
print("P[t]   :", P)
print("P[t]%k :", [p % k for p in P])

t      : [0, 1, 2, 3, 4, 5]
P[t]   : [0, 23, 25, 29, 35, 42]
P[t]%k : [0, 5, 1, 5, 5, 0]


## 5. Cách 1 — Brute Force / baseline: thử mọi đoạn

### 5.1. Ý tưởng

Duyệt mọi cặp `(i, j)` với `j > i` (độ dài ≥ 2). Với mỗi cặp, cộng các phần tử `nums[i..j]` rồi kiểm tra `% k == 0`. Có `O(n²)` cặp, mỗi cặp cộng `O(n)` phần tử, tổng `O(n³)`.

In [9]:
def brute_n3(nums, k):
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            total = 0
            for t in range(i, j + 1):      # cộng lại từ đầu mỗi lần
                total += nums[t]
            if total % k == 0:
                return True
    return False

for nums, k, expected in examples:
    got = brute_n3(nums, k)
    print(nums, k, "->", got)
    assert got == expected

[23, 2, 4, 6, 7] 6 -> True
[23, 2, 6, 4, 7] 6 -> True
[23, 2, 6, 4, 7] 13 -> False
[5] 5 -> False
[0, 0] 100 -> True


### 5.2. Cải tiến nhỏ: giữ "tổng chạy" → `O(n²)`

Khi cố định `i` và tăng `j`, tổng `nums[i..j]` chỉ hơn tổng `nums[i..j-1]` đúng một phần tử `nums[j]`. Vậy **cộng dồn** thay vì cộng lại từ đầu: bỏ vòng trong cùng.

In [10]:
def brute_n2(nums, k):
    n = len(nums)
    for i in range(n):
        total = nums[i]
        for j in range(i + 1, n):
            total += nums[j]                # tổng nums[i..j]
            if total % k == 0:
                return True
    return False

for nums, k, expected in examples:
    assert brute_n2(nums, k) == expected
print("brute_n2 khớp 5 ví dụ")

brute_n2 khớp 5 ví dụ


### 5.3. Độ phức tạp và nút thắt

- `brute_n3`: Time `O(n³)`, Space `O(1)`.
- `brute_n2`: Time `O(n²)`, Space `O(1)`.

Với `n = 10^5` thì `n² = 10^10` phép tính — quá chậm. Nút thắt: ta **tính lại gần như cùng một thông tin** (tổng các đoạn chồng lên nhau) và **thử từng cặp**. Hãy xem thời gian thực tế ở trường hợp xấu nhất (không có đoạn nào tốt nên phải duyệt hết):

In [11]:
n = 3000
worst = [1] * n          # k = n + 7 lớn hơn tổng nên không đoạn nào chia hết
t0 = time.perf_counter()
print(brute_n2(worst, 10**9 + 7))
print(f"brute_n2 với n = {n}: {time.perf_counter() - t0:.2f} giây (n = 10^5 sẽ chậm hơn ~1100 lần)")

False
brute_n2 với n = 3000: 0.24 giây (n = 10^5 sẽ chậm hơn ~1100 lần)


## 6. Ý tưởng thuật toán / bước đột phá

Ba bước suy luận:

**Bước 1 — biến tổng đoạn thành hiệu hai tiền tố.** Tổng `nums[i..j] = P[j+1] - P[i]`.

**Bước 2 — biến "chia hết" thành "cùng số dư".** `(P[j+1] - P[i]) % k == 0` khi và chỉ khi `P[j+1] % k == P[i] % k` (tính chất 2 ở mục 3.3, và chiều ngược lại cũng đúng vì hiệu chia hết cho `k` nghĩa là hai số dư bằng nhau).

**Bước 3 — bài toán trở thành tìm cặp chỉ số.** Cần hai chỉ số `a < b` trong `P` với `P[a] % k == P[b] % k` và đoạn giữa dài ít nhất 2, tức `b - a >= 2` (vì độ dài đoạn là `b - a`).

Từ đó: duyệt `b` từ trái sang phải, nhớ **chỉ số đầu tiên** `a` mà mỗi số dư xuất hiện. Vì sao là *đầu tiên*? Với cùng số dư, chỉ số nhỏ nhất cho khoảng cách `b - a` **lớn nhất** — nếu chỉ số sớm nhất còn không đạt khoảng cách 2 thì các chỉ số muộn hơn càng không đạt. Cần "mồi" số dư `0` ở chỉ số `0` (tiền tố rỗng `P[0] = 0`) để bắt các đoạn bắt đầu từ đầu mảng.

**Bonus — ngăn kéo:** có `n + 1` tiền tố nhưng chỉ `k` số dư. Nếu `n + 1 > 2k`, tức `n >= 2k`, thì có số dư xuất hiện ≥ 3 lần, trong 3 chỉ số khác nhau thì hai chỉ số xa nhất cách nhau ≥ 2 → **chắc chắn `True`** (chi tiết ở cách 5).

## 7. Các cách tiếp theo

Tổng cộng có 5 cách: Cách 1 (vét cạn, mục 5) và bốn cách dưới đây. Mỗi cách có: giải thích từng đoạn → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Tổng tiền tố theo modulo + `dict` chỉ số đầu tiên (`O(n)`)

Giải thích từng đoạn:

1. `d = {0: -1}`: số dư `0` đã xuất hiện ở "chỉ số -1" (tiền tố rỗng, trước phần tử `0`). Dùng chỉ số theo **phần tử**: sau khi cộng `nums[i]` ta được số dư `s`, đoạn kết thúc ở `i`; đoạn bắt đầu sau chỉ số `d[s]` nên có độ dài `i - d[s]`.
2. `s = (s + x) % k`: cập nhật số dư tiền tố, luôn giữ `s` nhỏ.
3. Nếu `s` chưa có trong `d`: ghi `d[s] = i` (lần đầu).
4. Nếu `s` đã có và `i - d[s] > 1` (độ dài đoạn ≥ 2): trả `True`. Nếu `i - d[s] == 1` (độ dài 1) thì **không** trả về và **không** ghi đè `d[s]`.
5. Hết vòng lặp: `False`.

In [12]:
def check_dict_demo(nums, k):
    d = {0: -1}                  # số dư 0 xuất hiện ở "chỉ số -1"
    s = 0
    for i, x in enumerate(nums):
        s = (s + x) % k
        if s not in d:
            d[s] = i                         # lần đầu thấy số dư này
        elif i - d[s] > 1:                   # đoạn d[s]+1 .. i dài >= 2
            print(f"  tìm thấy: nums[{d[s] + 1}..{i}] = {nums[d[s] + 1:i + 1]}")
            return True
    return False

for nums, k, expected in examples:
    print(nums, k)
    assert check_dict_demo(nums, k) == expected

[23, 2, 4, 6, 7] 6
  tìm thấy: nums[1..2] = [2, 4]
[23, 2, 6, 4, 7] 6
  tìm thấy: nums[1..3] = [2, 6, 4]
[23, 2, 6, 4, 7] 13
[5] 5
[0, 0] 100
  tìm thấy: nums[0..1] = [0, 0]


In [13]:
class Sol2:
    """Cách 2: prefix mod + dict chỉ số đầu tiên."""
    def checkSubarraySum(self, nums: List[int], k: int) -> bool:
        d = {0: -1}
        s = 0
        for i, x in enumerate(nums):
            s = (s + x) % k
            if s not in d:
                d[s] = i
            elif i - d[s] > 1:
                return True
        return False

for nums, k, expected in examples:
    assert Sol2().checkSubarraySum(nums, k) == expected
print("Sol2 OK")

Sol2 OK


**Độ phức tạp:** Time `O(n)` (mỗi phần tử một lần, thao tác dict `O(1)` trung bình). Space `O(min(n, k))` vì dict có tối đa `min(n+1, k)` khoá (số dư khác nhau).

### 7.2. Cách 3 — `set` "trễ một nhịp" (không cần nhớ chỉ số)

Ý tưởng khác nhẹ: điều kiện `b - a >= 2` có thể thay bằng "chỉ so với những tiền tố **cách xa ít nhất 2**". Ta giữ tập `seen` chứa số dư của các tiền tố `P[0..b-2]`. Tại bước xử lý phần tử thứ `i` (tiền tố mới `P[i+1]`), `seen` phải chứa `P[0..i-1]`. Tiền tố liền trước `P[i]` được lưu ở biến `prev` và chỉ được **thêm vào `seen` sau khi đã kiểm tra** — đó là "trễ một nhịp".

1. `seen = set()`, `prev = 0` (số dư của `P[0]`), `s = 0`.
2. Với mỗi `x`: cập nhật `s = (s + x) % k`.
3. Nếu `s in seen` → có `P[i+1]` cùng số dư với một `P[m]`, `m <= i-1` → `True`.
4. Thêm `prev` vào `seen` rồi `prev = s`.

In [14]:
def check_set_demo(nums, k):
    seen, prev, s = set(), 0, 0
    for i, x in enumerate(nums):
        s = (s + x) % k
        print(f"  i={i} x={x:>2} s={s} seen={sorted(seen)} prev={prev}")
        if s in seen:
            return True
        seen.add(prev)                       # trễ một nhịp: prev thành "đủ xa" cho lần sau
        prev = s
    return False

print(check_set_demo([23, 2, 6, 4, 7], 13))
print(check_set_demo([23, 2, 4, 6, 7], 6))

  i=0 x=23 s=10 seen=[] prev=0
  i=1 x= 2 s=12 seen=[0] prev=10
  i=2 x= 6 s=5 seen=[0, 10] prev=12
  i=3 x= 4 s=9 seen=[0, 10, 12] prev=5
  i=4 x= 7 s=3 seen=[0, 5, 10, 12] prev=9
False
  i=0 x=23 s=5 seen=[] prev=0
  i=1 x= 2 s=1 seen=[0] prev=5
  i=2 x= 4 s=5 seen=[0, 5] prev=1
True


In [15]:
class Sol3:
    """Cách 3: set các số dư tiền tố cách xa >= 2."""
    def checkSubarraySum(self, nums: List[int], k: int) -> bool:
        seen = set()
        prev = 0
        s = 0
        for x in nums:
            s = (s + x) % k
            if s in seen:
                return True
            seen.add(prev)
            prev = s
        return False

for nums, k, expected in examples:
    assert Sol3().checkSubarraySum(nums, k) == expected
print("Sol3 OK")

Sol3 OK


**Độ phức tạp:** Time `O(n)`, Space `O(min(n, k))`. Ưu điểm: không cần lưu chỉ số, chỉ cần `set`; điều kiện độ dài được "mã hoá" vào thời điểm thêm phần tử.

### 7.3. Cách 4 — DP trên tập các số dư của "đoạn kết thúc tại `i`"

Một góc nhìn khác hẳn, không dùng tiền tố: gọi `ends` là tập các **số dư tổng** của mọi mảng con **kết thúc ở phần tử trước** (độ dài ≥ 1). Khi gặp `x`:

- Mọi đoạn kết thúc ở `x` và dài ≥ 2 là "đoạn cũ kéo dài thêm `x`", có số dư `(r + x) % k` với `r` thuộc `ends`. Nếu có số dư `0` → `True`.
- Tập mới `ends = {(r + x) % k} ∪ {x % k}` (thêm đoạn chỉ gồm `x`).

Số phần tử của `ends` luôn `<= min(i+1, k)`, nên tổng thời gian `O(n · min(n, k))` — tốt khi `k` nhỏ, tệ khi `k` lớn.

In [16]:
def check_dp_demo(nums, k):
    ends = set()
    for i, x in enumerate(nums):
        extended = {(r + x) % k for r in ends}
        print(f"  i={i} x={x:>2} ends cũ={sorted(ends)} -> kéo dài={sorted(extended)}")
        if 0 in extended:
            return True
        ends = extended | {x % k}
    return False

print(check_dp_demo([23, 2, 4, 6, 7], 6))

  i=0 x=23 ends cũ=[] -> kéo dài=[]
  i=1 x= 2 ends cũ=[5] -> kéo dài=[1]
  i=2 x= 4 ends cũ=[1, 2] -> kéo dài=[0, 5]
True


In [17]:
class Sol4:
    """Cách 4: DP tập số dư của các đoạn kết thúc tại vị trí hiện tại."""
    def checkSubarraySum(self, nums: List[int], k: int) -> bool:
        ends = set()
        for x in nums:
            extended = {(r + x) % k for r in ends}
            if 0 in extended:
                return True
            ends = extended | {x % k}
        return False

for nums, k, expected in examples:
    assert Sol4().checkSubarraySum(nums, k) == expected
print("Sol4 OK")

Sol4 OK


**Độ phức tạp:** Time `O(n · min(n, k))`, Space `O(min(n, k))`. Chậm hơn Cách 2 khi `k` lớn, nhưng là bước đệm tốt để hiểu "trạng thái = số dư".

### 7.4. Cách 5 — Định lý ngăn kéo làm lối tắt + Cách 2

**Phát biểu:** nếu `len(nums) >= 2k` thì **luôn có mảng con tốt**.

**Chứng minh:** có `n + 1` tiền tố `P[0..n]` và chỉ `k` giá trị số dư. Nếu `n + 1 >= 2k + 1`, tức `n >= 2k`, theo ngăn kéo có một số dư xuất hiện ≥ 3 lần, tại các chỉ số `a < b < c`. Khi đó `c - a >= 2` nên đoạn `nums[a..c-1]` dài ≥ 2 và tổng chia hết cho `k`. ∎

Vậy với `k` nhỏ (`k <= n/2`) ta trả `True` ngay trong `O(1)`; các trường hợp còn lại dùng Cách 2. Đây chỉ là **tối ưu nhánh sớm**, độ phức tạp tệ nhất vẫn `O(n)`, nhưng nó cho ta thấy bài toán khó thật sự chỉ khi `k > n/2`.

In [18]:
class Sol5:
    """Cách 5: lối tắt ngăn kéo rồi Cách 2."""
    def checkSubarraySum(self, nums: List[int], k: int) -> bool:
        if len(nums) >= 2 * k:                # ngăn kéo: chắc chắn có
            return True
        d = {0: -1}
        s = 0
        for i, x in enumerate(nums):
            s = (s + x) % k
            if s not in d:
                d[s] = i
            elif i - d[s] > 1:
                return True
        return False

for nums, k, expected in examples:
    assert Sol5().checkSubarraySum(nums, k) == expected
print("Sol5 OK")
print("k=1, n=2 :", Sol5().checkSubarraySum([3, 4], 1), "(vì 2 >= 2*1)")
print("k=3, n=6 :", Sol5().checkSubarraySum([1, 1, 1, 1, 1, 1], 3), "(vì 6 >= 2*3)")

Sol5 OK
k=1, n=2 : True (vì 2 >= 2*1)
k=3, n=6 : True (vì 6 >= 2*3)


**Độ phức tạp:** Time `O(n)` (nhánh sớm `O(1)`), Space `O(min(n, k))`.

## 8. TRACE từng bước

### 8.1. Trace Cách 2 (dict chỉ số đầu tiên)

Ví dụ `nums = [23, 2, 6, 4, 7]`, `k = 13` → kết quả `False`. Bảng trạng thái (đầu: `d = {0: -1}`, `s = 0`):

| `i` | `x` | `s` mới `= (s+x) % 13` | `s` có trong `d`? | hành động | `d` sau bước |
|---|---|---|---|---|---|
| 0 | 23 | 10 | không | ghi `d[10] = 0` | `{0:-1, 10:0}` |
| 1 | 2 | 12 | không | ghi `d[12] = 1` | `{0:-1, 10:0, 12:1}` |
| 2 | 6 | 5 | không | ghi `d[5] = 2` | `{…, 5:2}` |
| 3 | 4 | 9 | không | ghi `d[9] = 3` | `{…, 9:3}` |
| 4 | 7 | 3 | không | ghi `d[3] = 4` | `{…, 3:4}` |

Không có số dư nào lặp lại → `False`.

In [19]:
def trace_dict(nums, k):
    d = {0: -1}
    s = 0
    print(f"nums={nums}, k={k}, d ban đầu={d}")
    for i, x in enumerate(nums):
        s = (s + x) % k
        if s not in d:
            d[s] = i
            action = f"lần đầu thấy dư {s} -> ghi d[{s}] = {i}"
        elif i - d[s] > 1:
            action = f"dư {s} đã thấy ở {d[s]}, độ dài = {i - d[s]} >= 2 -> TRẢ VỀ True (đoạn nums[{d[s] + 1}..{i}])"
            print(f"i={i} x={x:>2} s={s:>2} | {action}")
            return True
        else:
            action = f"dư {s} đã thấy ở {d[s]}, độ dài = {i - d[s]} < 2 -> bỏ qua (KHÔNG ghi đè)"
        print(f"i={i} x={x:>2} s={s:>2} | {action} | d={d}")
    print("hết mảng -> False")
    return False

trace_dict([23, 2, 6, 4, 7], 13)

nums=[23, 2, 6, 4, 7], k=13, d ban đầu={0: -1}
i=0 x=23 s=10 | lần đầu thấy dư 10 -> ghi d[10] = 0 | d={0: -1, 10: 0}
i=1 x= 2 s=12 | lần đầu thấy dư 12 -> ghi d[12] = 1 | d={0: -1, 10: 0, 12: 1}
i=2 x= 6 s= 5 | lần đầu thấy dư 5 -> ghi d[5] = 2 | d={0: -1, 10: 0, 12: 1, 5: 2}
i=3 x= 4 s= 9 | lần đầu thấy dư 9 -> ghi d[9] = 3 | d={0: -1, 10: 0, 12: 1, 5: 2, 9: 3}
i=4 x= 7 s= 3 | lần đầu thấy dư 3 -> ghi d[3] = 4 | d={0: -1, 10: 0, 12: 1, 5: 2, 9: 3, 3: 4}
hết mảng -> False


False

### 8.2. Trace trường hợp `True` có "bẫy độ dài 1"

`nums = [4, 4, 1, 2]`, `k = 4`? Tiền tố số dư: `0, 0, 0, 1, 3`. Số dư `4 % 4 = 0` ngay ở `i = 0` trùng `d[0] = -1` với độ dài `0 - (-1) = 1` — **chưa đủ** (mảng con `[4]` đơn lẻ). Phải đợi `i = 1` (độ dài 2). Hãy chạy trace để thấy cách thuật toán bỏ qua rồi vẫn tìm ra:

In [20]:
trace_dict([4, 4, 1, 2], 4)
print()
trace_dict([5], 5)

nums=[4, 4, 1, 2], k=4, d ban đầu={0: -1}
i=0 x= 4 s= 0 | dư 0 đã thấy ở -1, độ dài = 1 < 2 -> bỏ qua (KHÔNG ghi đè) | d={0: -1}
i=1 x= 4 s= 0 | dư 0 đã thấy ở -1, độ dài = 2 >= 2 -> TRẢ VỀ True (đoạn nums[0..1])

nums=[5], k=5, d ban đầu={0: -1}
i=0 x= 5 s= 0 | dư 0 đã thấy ở -1, độ dài = 1 < 2 -> bỏ qua (KHÔNG ghi đè) | d={0: -1}
hết mảng -> False


False

Quan sát: ở `[5]`, `k = 5`, số dư `0` trùng nhưng độ dài 1 nên trả `False` đúng. Việc **không ghi đè** `d[0] = -1` là quan trọng: nếu ta ghi `d[0] = 0` thì ở `i = 1` ta mất cơ hội so khoảng cách từ `-1`.

### 8.3. Trace Cách 3 (set trễ một nhịp) bằng hàm có bảng

Cột `seen` là những số dư tiền tố đủ xa (`P[0..i-1]`) *trước* khi kiểm tra.

In [21]:
def trace_set(nums, k):
    seen, prev, s = set(), 0, 0
    print("i  x   s  prev  seen(trước)   s in seen?")
    for i, x in enumerate(nums):
        s = (s + x) % k
        hit = s in seen
        print(f"{i}  {x:>2}  {s:>2}  {prev:>3}   {str(sorted(seen)):<12}  {hit}")
        if hit:
            return True
        seen.add(prev)
        prev = s
    return False

print(trace_set([23, 2, 4, 6, 7], 6))
print(trace_set([1, 2, 3], 7))

i  x   s  prev  seen(trước)   s in seen?
0  23   5    0   []            False
1   2   1    5   [0]           False
2   4   5    1   [0, 5]        True
True
i  x   s  prev  seen(trước)   s in seen?
0   1   1    0   []            False
1   2   3    1   [0]           False
2   3   6    3   [0, 1]        False
False


### 8.4. Trace Cách 4 (DP tập số dư)

Mỗi dòng cho thấy `ends` trước, tập sau khi kéo dài, và có `0` hay không.

In [22]:
def trace_dp(nums, k):
    ends = set()
    for i, x in enumerate(nums):
        extended = {(r + x) % k for r in ends}
        found = 0 in extended
        new_ends = extended | {x % k}
        print(f"i={i} x={x:>2} | ends cũ={str(sorted(ends)):<14} -> kéo dài={str(sorted(extended)):<14} có 0? {found} | ends mới={sorted(new_ends)}")
        if found:
            return True
        ends = new_ends
    return False

print(trace_dp([23, 2, 6, 4, 7], 6))

i=0 x=23 | ends cũ=[]             -> kéo dài=[]             có 0? False | ends mới=[5]
i=1 x= 2 | ends cũ=[5]            -> kéo dài=[1]            có 0? False | ends mới=[1, 2]
i=2 x= 6 | ends cũ=[1, 2]         -> kéo dài=[1, 2]         có 0? False | ends mới=[0, 1, 2]
i=3 x= 4 | ends cũ=[0, 1, 2]      -> kéo dài=[0, 4, 5]      có 0? True | ends mới=[0, 4, 5]
True


## 9. Code nộp LeetCode

Dán đoạn dưới vào ô code của LeetCode (trên LeetCode đã có sẵn `List` nên không cần `import`; ở notebook ta đã import ở đầu).

In [23]:
class Solution:
    def checkSubarraySum(self, nums: List[int], k: int) -> bool:
        d = {0: -1}              # số dư 0 xuất hiện ở "chỉ số -1" (tiền tố rỗng)
        s = 0
        for i, x in enumerate(nums):
            s = (s + x) % k
            if s not in d:
                d[s] = i         # chỉ ghi lần đầu tiên
            elif i - d[s] > 1:   # đoạn d[s]+1 .. i có độ dài >= 2
                return True
        return False


for nums, k, expected in examples:
    assert Solution().checkSubarraySum(nums, k) == expected
print("Solution chạy đúng 5 ví dụ")

Solution chạy đúng 5 ví dụ


## 10. Vì sao thuật toán đúng

Đặt `S_i` là số dư của `P[i+1]` (tổng `nums[0..i]` chia `k`), và `S_{-1} = 0` (tiền tố rỗng, tương ứng `d[0] = -1`).

**Bổ đề.** Với `a < b` (chỉ số theo phần tử, `a >= -1`), tổng `nums[a+1..b]` chia hết cho `k` khi và chỉ khi `S_a == S_b`. *Chứng minh:* `sum(nums[a+1..b]) = P[b+1] - P[a+1]`; hiệu của hai số chia hết cho `k` khi và chỉ khi hai số cùng số dư. ∎

Đoạn `nums[a+1..b]` có độ dài `b - a`, nên là đoạn tốt khi `S_a == S_b` **và** `b - a >= 2`.

**Bất biến của vòng lặp.** Trước khi xử lý phần tử `i`, với mỗi số dư `r` đã từng xuất hiện trong `S_{-1..i-1}`, `d[r]` là **chỉ số nhỏ nhất** `a` thoả `S_a = r`.

- *Khởi tạo:* `d = {0: -1}` đúng với `S_{-1} = 0`.
- *Duy trì:* nếu `S_i` chưa có trong `d`, ta thêm `d[S_i] = i` (nhỏ nhất vì là lần đầu). Nếu đã có thì **không ghi đè**, nên `d[S_i]` vẫn là chỉ số nhỏ nhất.

**Tính đúng (không nhầm lẫn):** khi trả `True` ta có `S_i == S_{d[S_i]}` và `i - d[S_i] > 1` → bổ đề cho ta một đoạn tốt thật.

**Tính đầy đủ (không bỏ sót):** giả sử tồn tại đoạn tốt `nums[a+1..b]`, tức `S_a == S_b`, `b - a >= 2`. Tại bước `b`, `d[S_b]` là chỉ số nhỏ nhất `a0 <= a` có số dư đó, nên `b - a0 >= b - a >= 2` và điều kiện `i - d[s] > 1` thoả → trả `True` (hoặc đã trả `True` sớm hơn). Nếu thuật toán chạy hết mà không trả `True`, thì không có đoạn tốt nào. ∎

Hai chi tiết nhỏ cần nhớ:

- **Vì sao mồi `-1`:** nếu không, đoạn bắt đầu từ đầu mảng (như `[23,2,6,4,7]` với `k=6`, tổng 42) bị bỏ sót.
- **Vì sao dùng `elif` chứ không `if`:** khi `s` chưa có trong `d` ta chỉ ghi; khi đã có ta chỉ so độ dài — hai nhánh loại trừ nhau.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1a. Vét cạn tam trùng | `O(n³)` | `O(1)` |
| 1b. Vét cạn với tổng chạy | `O(n²)` | `O(1)` |
| 2. Prefix mod + dict | `O(n)` | `O(min(n, k))` |
| 3. Set trễ một nhịp | `O(n)` | `O(min(n, k))` |
| 4. DP tập số dư | `O(n · min(n, k))` | `O(min(n, k))` |
| 5. Ngăn kéo + Cách 2 | `O(n)` | `O(min(n, k))` |

Với `n = 10^5`: `O(n)` chỉ cần ~`10^5` phép tính; `O(n²)` là `10^10`. Giới hạn tổng `≤ 2^31 - 1` chủ yếu để C++/Java khỏi tràn `int` khi cộng; Python có số nguyên lớn tuỳ ý nên không lo, và ta còn lấy `% k` mỗi bước nên `s` luôn `< k`.

Đo thực nghiệm: so Cách 2 (`O(n)`) với vét cạn `O(n²)` trên cùng mảng.

In [24]:
def timeit(fn, *args):
    t0 = time.perf_counter()
    res = fn(*args)
    return res, time.perf_counter() - t0

rng = random.Random(523)
for n in (500, 1000, 2000):
    arr = [1] * n                                     # tổng n < k: không có đoạn tốt, phải duyệt hết
    k = 10**9 + 7
    r1, t1 = timeit(brute_n2, arr, k)
    r2, t2 = timeit(Solution().checkSubarraySum, arr, k)
    assert r1 == r2
    print(f"n={n:5d}: brute_n2 {t1*1000:8.2f} ms | dict {t2*1000:7.3f} ms | kết quả {r1}")

big = [rng.randint(1, 10) for _ in range(10**5)]      # không có số 0 -> k lớn thì không có đoạn tốt
r, t = timeit(Solution().checkSubarraySum, big, 2**31 - 1)
print(f"n=10^5: dict {t*1000:.1f} ms | kết quả {r}")

n=  500: brute_n2     6.11 ms | dict   0.151 ms | kết quả False
n= 1000: brute_n2    25.98 ms | dict   0.161 ms | kết quả False


n= 2000: brute_n2   105.52 ms | dict   0.398 ms | kết quả False


n=10^5: dict 21.6 ms | kết quả False


## 12. Bộ test

### 12.1. Ví dụ đề và edge case

**Oracle độc lập:** dùng `itertools.combinations` để chọn mọi cặp chỉ số `(i, j)`, tính tổng bằng `sum(nums[i:j+1])` — không dùng tổng chạy, tổng tiền tố hay số dư giữa chừng. Ta gom các cách vào dict `solvers`.

In [25]:
def oracle(nums, k):
    n = len(nums)
    for i, j in combinations(range(n), 2):       # i < j  -> độ dài >= 2
        if sum(nums[i:j + 1]) % k == 0:
            return True
    return False

solvers = {
    "brute_n3": brute_n3,
    "brute_n2": brute_n2,
    "dict": lambda a, k: Sol2().checkSubarraySum(a, k),
    "set": lambda a, k: Sol3().checkSubarraySum(a, k),
    "dp": lambda a, k: Sol4().checkSubarraySum(a, k),
    "pigeon": lambda a, k: Sol5().checkSubarraySum(a, k),
    "leetcode": lambda a, k: Solution().checkSubarraySum(a, k),
}

for nums, k, expected in examples:
    assert oracle(nums, k) == expected
    for name, fn in solvers.items():
        assert fn(nums, k) == expected, (name, nums, k)
print("Ví dụ đề: OK cho cả 7 solver và oracle")

Ví dụ đề: OK cho cả 7 solver và oracle


In [26]:
edge_cases = [
    ([0], 1, False, "một phần tử"),
    ([0, 0], 1, True, "hai số 0"),
    ([0, 0], 2**31 - 1, True, "tổng 0 với k cực lớn"),
    ([1, 1], 2, True, "đúng hai phần tử tạo bội"),
    ([1, 2], 3, True, "đúng hai phần tử"),
    ([1, 2], 4, False, "hai phần tử không chia hết"),
    ([5], 5, False, "một phần tử là bội nhưng độ dài 1"),
    ([5, 0], 5, True, "5 + 0"),
    ([0, 5], 5, True, "0 + 5"),
    ([3, 4], 1, True, "k = 1 luôn chia hết"),
    ([1, 2, 3], 5, True, "2 + 3 = 5 (không bắt đầu ở đầu)"),
    ([1, 2, 3], 6, True, "cả mảng = 6"),
    ([1, 2, 3], 7, False, "không đoạn nào"),
    ([7, 1, 5], 7, False, "[7] đơn lẻ không tính; 7+1=8, 1+5=6, 13"),
    ([6, 6], 6, True, "6 + 6 = 12"),
    ([10**9, 10**9, 10**9], 2 * 10**9, True, "tổng hai phần tử = 2*10^9 (chỉ để thử biên lớn)"),
    ([2**30, 2**30 - 1], 2**31 - 1, True, "tổng đúng 2^31 - 1"),
    ([2**30, 2**30 - 1], 2**31 - 2, False, "tổng 2^31-1 không chia hết 2^31-2"),
]
for nums, k, expected, note in edge_cases:
    assert oracle(nums, k) == expected, ("oracle", note)
    for name, fn in solvers.items():
        assert fn(nums, k) == expected, (name, note)
print(f"{len(edge_cases)} edge case: OK")

18 edge case: OK


### 12.2. Phản ví dụ cho các ý tưởng sai

Những "ý tưởng nghe hợp lý" nhưng sai — ta giữ lại để chắc chắn lời giải đúng không mắc phải:

- "Chỉ cần thấy một số dư lặp lại là xong" (bỏ điều kiện độ dài ≥ 2): sai với `[5]`, `k = 5` (số dư `0` lặp lại nhưng đoạn chỉ có 1 phần tử) và với `[1, 0]`, `k = 3` (dư tiền tố `0,1,1` lặp ở `i=1` với độ dài 1, thực tế `1 + 0 = 1` không chia hết 3 nên đáp án là `False`).
- "Ghi đè `d[s] = i` mỗi lần": sai với `[1, 0, 0]`, `k = 3` (đáp án đúng là `True` vì `0 + 0 = 0`). Ta kiểm tra bằng code dưới đây.

In [27]:
def wrong_no_length(nums, k):                 # bỏ điều kiện độ dài
    d = {0: -1}
    s = 0
    for i, x in enumerate(nums):
        s = (s + x) % k
        if s in d:
            return True
        d[s] = i
    return False

def wrong_overwrite(nums, k):                 # ghi đè chỉ số mỗi lần
    d = {0: -1}
    s = 0
    for i, x in enumerate(nums):
        s = (s + x) % k
        if s in d and i - d[s] > 1:
            return True
        d[s] = i
    return False

counter_examples = [([5], 5), ([1, 0], 3), ([1, 0, 0], 3), ([2, 0, 0], 5), ([4, 4, 1, 2], 4)]
for nums, k in counter_examples:
    print(f"{nums!s:<14} k={k}: đúng={oracle(nums, k)!s:<5} | bỏ độ dài={wrong_no_length(nums, k)!s:<5} | ghi đè={wrong_overwrite(nums, k)!s:<5}")
assert wrong_no_length([5], 5) != oracle([5], 5)
assert wrong_no_length([1, 0], 3) != oracle([1, 0], 3)
assert wrong_overwrite([1, 0, 0], 3) != oracle([1, 0, 0], 3)

[5]            k=5: đúng=False | bỏ độ dài=True  | ghi đè=False
[1, 0]         k=3: đúng=False | bỏ độ dài=True  | ghi đè=False
[1, 0, 0]      k=3: đúng=True  | bỏ độ dài=True  | ghi đè=False
[2, 0, 0]      k=5: đúng=True  | bỏ độ dài=True  | ghi đè=False
[4, 4, 1, 2]   k=4: đúng=True  | bỏ độ dài=True  | ghi đè=False


Bảng cho thấy `wrong_no_length` sai ở `[5]` và `[1, 0]`, còn "ghi đè" sai ở `[1, 0, 0]` nhưng vẫn đúng với nhiều ví dụ khác. Ta thử tìm phản ví dụ **nhỏ nhất** cho "ghi đè" bằng cách duyệt tự động các mảng ngắn:

In [28]:
from itertools import product
found = None
for n in range(2, 6):
    for nums in product(range(4), repeat=n):
        for k in range(1, 8):
            if wrong_overwrite(list(nums), k) != oracle(list(nums), k):
                found = (list(nums), k)
                break
        if found:
            break
    if found:
        break
print("phản ví dụ nhỏ nhất cho 'ghi đè':", found, "| đúng:", oracle(*found), "| sai:", wrong_overwrite(*found))
assert found is not None

phản ví dụ nhỏ nhất cho 'ghi đè': ([0, 0], 1) | đúng: True | sai: False


Vì sao ghi đè sai: khi `i - d[s] == 1` ta ghi `d[s] = i` làm mất chỉ số *sớm hơn* — về sau không còn khoảng cách đủ xa.

### 12.3. Đối chiếu mọi cách bằng vét cạn toàn bộ input nhỏ

Duyệt **mọi** mảng độ dài `1..5` với giá trị `0..3` và `k` từ `1..7`, so sánh 7 solver với oracle.

In [29]:
checked = 0
for n in range(1, 6):
    for nums in product(range(4), repeat=n):
        nums = list(nums)
        for k in range(1, 8):
            expected = oracle(nums, k)
            for name, fn in solvers.items():
                assert fn(nums, k) == expected, (name, nums, k)
            checked += 1
print("đã kiểm", checked, "cặp (nums, k): tất cả khớp oracle")

đã kiểm 9548 cặp (nums, k): tất cả khớp oracle


### 12.4. Test ngẫu nhiên có seed (đúng ràng buộc)

Sinh `nums[i]` không âm, `k >= 1`, tổng không vượt `2^31 - 1`. Ba chế độ: số nhỏ + `k` nhỏ (nhiều số dư trùng), số nhỏ + `k` lớn (hầu như `False`), và số lớn với `n` nhỏ sao cho tổng vẫn ≤ `2^31 - 1`.

In [30]:
rng = random.Random(523)
LIM = 2**31 - 1

def gen(rng):
    mode = rng.choice(["small_k", "big_k", "big_vals"])
    if mode == "small_k":
        n = rng.randint(1, 12)
        nums = [rng.randint(0, 20) for _ in range(n)]
        k = rng.randint(1, 15)
    elif mode == "big_k":
        n = rng.randint(1, 12)
        nums = [rng.randint(0, 20) for _ in range(n)]
        k = rng.randint(30, LIM)
    else:
        n = rng.randint(1, 6)
        cap = LIM // n
        nums = [rng.randint(0, min(10**9, cap)) for _ in range(n)]
        k = rng.randint(1, LIM)
    assert sum(nums) <= LIM and 1 <= k <= LIM and all(0 <= x <= 10**9 for x in nums)
    return nums, k

trues = 0
for _ in range(4000):
    nums, k = gen(rng)
    expected = oracle(nums, k)
    trues += expected
    for name, fn in solvers.items():
        assert fn(nums, k) == expected, (name, nums, k)
print("4000 test ngẫu nhiên: OK;", trues, "ca True,", 4000 - trues, "ca False")

4000 test ngẫu nhiên: OK; 964 ca True, 3036 ca False


### 12.5. Test lớn (hiệu năng) và so sánh các cách `O(n)`

Với `n = 10^5` ta không dùng oracle (quá chậm) mà đối chiếu 4 cách `O(n)`/lối tắt với nhau và dựng trước kết quả biết chắc. Cách 4 (DP) chỉ chạy khi `k` nhỏ.

In [31]:
rng = random.Random(5230)
N = 10**5

# 1) tất cả bằng 1 và k = N + 1: tổng cả mảng = N < k, các tiền tố 0..N đều khác nhau -> False
arr = [1] * N
for fn in (solvers["dict"], solvers["set"], solvers["pigeon"], solvers["leetcode"]):
    assert fn(arr, N + 1) is False
# 2) thêm hai số 0 liền nhau ở cuối -> chắc chắn True (đoạn [0, 0])
arr2 = [1] * (N - 2) + [0, 0]
for fn in (solvers["dict"], solvers["set"], solvers["pigeon"], solvers["leetcode"]):
    assert fn(arr2, N + 5) is True
# 3) ngẫu nhiên lớn, nhiều k: so sánh dict / set / pigeon / leetcode
for _ in range(20):
    nums = [rng.randint(0, 10) for _ in range(N)]
    k = rng.choice([1, 2, 7, 1000, 10**5, 10**6, 2**31 - 1])
    answers = {solvers[x](nums, k) for x in ("dict", "set", "pigeon", "leetcode")}
    assert len(answers) == 1
# 4) k nhỏ: DP cũng phải khớp
for k in (1, 2, 3, 50):
    nums = [rng.randint(0, 10) for _ in range(2000)]
    assert solvers["dp"](nums, k) == solvers["dict"](nums, k)
print("test lớn: OK")

test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên "mồi" `{0: -1}`

**Code sai:**

In [32]:
class WrongSolution1:
    def checkSubarraySum(self, nums, k):
        d = {}                          # thiếu {0: -1}
        s = 0
        for i, x in enumerate(nums):
            s = (s + x) % k
            if s not in d:
                d[s] = i
            elif i - d[s] > 1:
                return True
        return False

print("sai  :", WrongSolution1().checkSubarraySum([23, 2, 6, 4, 7], 6), "(đúng phải True: cả mảng tổng 42)")
print("đúng :", Solution().checkSubarraySum([23, 2, 6, 4, 7], 6))
print("sai  :", WrongSolution1().checkSubarraySum([2, 4], 6), "(đúng phải True: 2 + 4 = 6)")

sai  : True (đúng phải True: cả mảng tổng 42)
đúng : True
sai  : False (đúng phải True: 2 + 4 = 6)


**Vì sao sai:** đoạn bắt đầu từ phần tử đầu tiên có tổng tiền tố `P[b+1]` chia hết `k`, tức số dư `0`, mà ta chưa từng "thấy" số dư `0` trước đó. `{0: -1}` biểu diễn tiền tố rỗng (tổng 0). **Code đúng:** `d = {0: -1}`.

### Lỗi 2: Ghi đè chỉ số đầu tiên

In [33]:
class WrongSolution2:
    def checkSubarraySum(self, nums, k):
        d = {0: -1}
        s = 0
        for i, x in enumerate(nums):
            s = (s + x) % k
            if s in d and i - d[s] > 1:
                return True
            d[s] = i                    # sai: luôn ghi đè, mất chỉ số sớm nhất
        return False

bad = [3, 0, 0]
k = 3
print(bad, k, "| sai:", WrongSolution2().checkSubarraySum(bad, k), "| đúng:", Solution().checkSubarraySum(bad, k))
mismatch = None
for nums in product(range(3), repeat=5):
    for kk in range(1, 6):
        if WrongSolution2().checkSubarraySum(list(nums), kk) != oracle(list(nums), kk):
            mismatch = (list(nums), kk)
            break
    if mismatch:
        break
print("phản ví dụ tự tìm:", mismatch)
assert mismatch is not None

[3, 0, 0] 3 | sai: False | đúng: True
phản ví dụ tự tìm: ([0, 0, 0, 0, 0], 1)


**Vì sao sai:** khi gặp lại số dư `s` mà khoảng cách chưa đủ 2, ta lỡ ghi `d[s] = i`, mất đi chỉ số **sớm hơn** (khoảng cách xa hơn). Về sau một lần xuất hiện nữa của `s` sẽ chỉ so với chỉ số muộn. **Code đúng:** chỉ ghi `d[s] = i` khi `s` chưa có trong `d` (dùng `if ... not in d` / `elif`).

### Lỗi 3: Quên điều kiện độ dài ≥ 2

In [34]:
class WrongSolution3:
    def checkSubarraySum(self, nums, k):
        d = {0: -1}
        s = 0
        for i, x in enumerate(nums):
            s = (s + x) % k
            if s in d:                  # sai: không kiểm tra i - d[s] > 1
                return True
            d[s] = i
        return False

for nums, k in [([5], 5), ([1, 0], 3), ([1, 2], 4)]:
    print(nums, k, "| sai:", WrongSolution3().checkSubarraySum(nums, k), "| đúng:", Solution().checkSubarraySum(nums, k))

[5] 5 | sai: True | đúng: False
[1, 0] 3 | sai: True | đúng: False
[1, 2] 4 | sai: False | đúng: False


**Vì sao sai:** `[5]` có số dư tiền tố `0` trùng `d[0] = -1` nhưng đoạn chỉ có **một** phần tử, đề yêu cầu ≥ 2. **Code đúng:** thêm điều kiện `i - d[s] > 1`.

### Lỗi 4: Cộng tổng đầy đủ rồi chỉ kiểm tra `total % k` ở cuối (và bỏ qua đoạn giữa)

Một số bạn chỉ kiểm tra các **tiền tố** `P[t] % k == 0` (tức chỉ xét đoạn bắt đầu từ đầu mảng):

In [35]:
class WrongSolution4:
    def checkSubarraySum(self, nums, k):
        s = 0
        for i, x in enumerate(nums):
            s += x
            if i >= 1 and s % k == 0:   # chỉ xét đoạn bắt đầu từ đầu mảng
                return True
        return False

for nums, k in [([23, 2, 6, 4, 7], 6), ([7, 2, 4, 1], 6), ([1, 2, 3], 5)]:
    print(nums, k, "| sai:", WrongSolution4().checkSubarraySum(nums, k), "| đúng:", Solution().checkSubarraySum(nums, k))

[23, 2, 6, 4, 7] 6 | sai: True | đúng: True
[7, 2, 4, 1] 6 | sai: False | đúng: True
[1, 2, 3] 5 | sai: False | đúng: True


**Vì sao sai:** `[7,2,4,1]`, `k = 6` có đoạn `[2,4]` ở giữa (tổng 6) nhưng các tiền tố 9, 13, 14 không chia hết 6; tương tự `[1,2,3]`, `k = 5` có `2+3=5` nhưng tiền tố 3, 6 thì không. (Còn `[23,2,6,4,7]` được đoán đúng chỉ vì tiền tố 42 tình cờ chia hết 6.) **Code đúng:** xét **hiệu hai tiền tố** bất kỳ, thể hiện bằng `dict` số dư.

### Lỗi 5: Dùng `==` để so `sum` với `k` thay vì `% k`

In [36]:
def wrong_equals(nums, k):
    for i in range(len(nums)):
        total = nums[i]
        for j in range(i + 1, len(nums)):
            total += nums[j]
            if total == k:               # sai: bội của k chứ không phải đúng bằng k
                return True
    return False

print([23, 2, 6, 4, 7], 6, "| sai:", wrong_equals([23, 2, 6, 4, 7], 6), "| đúng:", Solution().checkSubarraySum([23, 2, 6, 4, 7], 6))
print([6, 6], 6, "| sai:", wrong_equals([6, 6], 6), "| đúng:", Solution().checkSubarraySum([6, 6], 6))

[23, 2, 6, 4, 7] 6 | sai: False | đúng: True
[6, 6] 6 | sai: False | đúng: True


**Vì sao sai:** đề hỏi **bội** của `k` (`0, k, 2k, 3k, ...`), không phải đúng bằng `k`. **Code đúng:** `total % k == 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"tổng đoạn liên tiếp" + "chia hết cho k" / "dư"** → nghĩ ngay **prefix sum modulo k** + bảng băm số dư.
- Thấy **"tổng đoạn = k"** (không chia hết) → prefix sum + `dict` `P[b] - k`, ví dụ LeetCode 560.
- Bài hỏi "**tồn tại**" thì chỉ cần `True/False`, có thể dừng sớm. Bài hỏi "**đếm**" thì cần `Counter` thay vì `dict` chỉ số đầu.
- Muốn đoạn **dài nhất** → lưu **chỉ số đầu tiên**; muốn đoạn **ngắn nhất** → lưu **chỉ số cuối cùng** (ghi đè mỗi lần).
- Ràng buộc "độ dài ≥ 2" thường chuyển thành `i - d[s] > 1` hoặc kỹ thuật "trễ một nhịp" như Cách 3.
- Khi số dư lặp nhiều, nghĩ **nguyên lý ngăn kéo**: `n >= 2k` ⇒ chắc chắn tồn tại.
- Tập `k` ngăn, `n` vật: nếu `n` lớn hơn `k` thì **số dư chắc chắn trùng** — đây là cách bài này hỏi "bắt buộc phải có không".
- Luôn thử tay: `[x]` một phần tử, `[0,0]`, `k = 1`, `k` lớn hơn tổng.

## 15. Biến thể

### 15.1. Bỏ ràng buộc: `k` có thể bằng 0 hoặc âm, số có thể âm (đề cũ)

Phiên bản đề cũ cho phép `k = 0` (khi đó "bội của 0" chỉ là `0`, tức đoạn có tổng đúng bằng 0) và số âm. Lúc đó dùng `s % k` nếu `k != 0`, còn không thì giữ nguyên tổng. Python `%` với `k` âm vẫn cho số dư nhất quán trong cùng một `k`, nên cùng ý tưởng hoạt động.

In [37]:
def check_general(nums, k):
    d = {0: -1}
    s = 0
    for i, x in enumerate(nums):
        s += x
        key = s % k if k != 0 else s
        if key not in d:
            d[key] = i
        elif i - d[key] > 1:
            return True
    return False

def oracle_general(nums, k):
    for i, j in combinations(range(len(nums)), 2):
        t = sum(nums[i:j + 1])
        if (t == 0) if k == 0 else (t % k == 0):
            return True
    return False

print(check_general([0, 0], 0), check_general([1, 0], 0), check_general([3, -3, 5], 0))
rng = random.Random(15)
for _ in range(3000):
    n = rng.randint(1, 8)
    nums = [rng.randint(-6, 6) for _ in range(n)]
    k = rng.randint(-7, 7)
    assert check_general(nums, k) == oracle_general(nums, k), (nums, k)
print("biến thể 15.1 khớp oracle trên 3000 test")

True False True
biến thể 15.1 khớp oracle trên 3000 test


### 15.2. Trả về cặp chỉ số của một đoạn tốt

Đổi `return True` thành `return (d[s] + 1, i)` (chỉ số đầu và cuối, tính cả hai đầu), còn không thì `None`.

In [38]:
def find_good(nums, k):
    d = {0: -1}
    s = 0
    for i, x in enumerate(nums):
        s = (s + x) % k
        if s not in d:
            d[s] = i
        elif i - d[s] > 1:
            return d[s] + 1, i
    return None

print(find_good([23, 2, 4, 6, 7], 6))
print(find_good([23, 2, 6, 4, 7], 13))
rng = random.Random(152)
for _ in range(2000):
    n = rng.randint(1, 10)
    nums = [rng.randint(0, 9) for _ in range(n)]
    k = rng.randint(1, 12)
    res = find_good(nums, k)
    assert (res is not None) == oracle(nums, k)
    if res:
        a, b = res
        assert b - a >= 1 and sum(nums[a:b + 1]) % k == 0
print("find_good: đoạn trả về luôn hợp lệ")

(1, 2)
None
find_good: đoạn trả về luôn hợp lệ


### 15.3. Đếm số mảng con tốt (độ dài ≥ 2)

Với mỗi tiền tố mới, cộng số tiền tố **cũ cách xa ít nhất 2** có cùng số dư. Dùng `Counter` và kỹ thuật "trễ một nhịp" như Cách 3.

In [39]:
def count_good(nums, k):
    cnt = Counter()
    prev, s, ans = 0, 0, 0
    for x in nums:
        s = (s + x) % k
        ans += cnt[s]               # các tiền tố P[0..i-1]
        cnt[prev] += 1              # prev = P[i] chỉ đủ xa cho lần sau
        prev = s
    return ans

def count_good_oracle(nums, k):
    return sum(1 for i, j in combinations(range(len(nums)), 2) if sum(nums[i:j + 1]) % k == 0)

print(count_good([23, 2, 4, 6, 7], 6), count_good([0, 0, 0], 5))
rng = random.Random(153)
for _ in range(2000):
    n = rng.randint(1, 10)
    nums = [rng.randint(0, 9) for _ in range(n)]
    k = rng.randint(1, 10)
    assert count_good(nums, k) == count_good_oracle(nums, k)
print("count_good khớp oracle")

3 3
count_good khớp oracle


### 15.4. Các bài liên quan trên LeetCode (đọc thêm)

- **974. Subarray Sums Divisible by K:** đếm mảng con (mọi độ dài ≥ 1) có tổng chia hết `k` — `Counter` số dư tiền tố, có thể có số âm nên dùng `%` của Python.
- **560. Subarray Sum Equals K:** tổng bằng đúng `k`, dùng `Counter` của chính tổng tiền tố.
- **525. Contiguous Array:** đổi `0 → -1` rồi tìm đoạn dài nhất có tổng 0 (dict chỉ số đầu tiên).
- **1590. Make Sum Divisible by P:** bỏ đoạn ngắn nhất để tổng còn lại chia hết `p`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Vét cạn `O(n³)` | thử mọi đoạn, cộng lại từ đầu | `O(n³)` | `O(1)` | chỉ để hiểu đề / làm oracle |
| 1b. Vét cạn `O(n²)` | thử mọi đoạn, cộng dồn | `O(n²)` | `O(1)` | `n <= 3000`, oracle nhanh hơn |
| 2. Prefix mod + dict | cùng số dư, nhớ chỉ số đầu | `O(n)` | `O(min(n,k))` | **cách nộp bài chuẩn** |
| 3. Set trễ một nhịp | cùng số dư, chỉ so với tiền tố đủ xa | `O(n)` | `O(min(n,k))` | khi không cần chỉ số; dễ mở rộng sang đếm |
| 4. DP tập số dư | trạng thái = số dư các đoạn kết thúc tại `i` | `O(n·min(n,k))` | `O(min(n,k))` | `k` nhỏ, muốn học góc nhìn DP |
| 5. Ngăn kéo + Cách 2 | `n >= 2k` ⇒ `True` | `O(n)` | `O(min(n,k))` | lối tắt, hiểu cấu trúc bài |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [40]:
print(Solution().checkSubarraySum([1, 2, 3], 5))
print(Solution().checkSubarraySum([1, 2, 3], 7))
print(Solution().checkSubarraySum([0], 1))
print(Solution().checkSubarraySum([6, 1], 6))

True
False
False
False


**Bài 2 (tính tay).** Với `nums = [4, 5, 0, 3]`, `k = 3`: viết dãy số dư tiền tố `S_{-1}, S_0, S_1, S_2, S_3` (bắt đầu bằng `0` cho tiền tố rỗng). Gán list vào `ans2` (đang là `None`).

In [41]:
ans2 = None      # TODO: list 5 số nguyên (số dư tiền tố, bắt đầu bằng 0)

**Bài 3 (điền TODO).** Hoàn thành hàm `shortest_good_length(nums, k)` trả về **độ dài ngắn nhất** của một mảng con tốt (độ dài ≥ 2), hoặc `0` nếu không có. Gợi ý: muốn đoạn **ngắn** nhất ta cần chỉ số **gần nhất** (mới nhất) của mỗi số dư, nhưng vẫn phải cách ít nhất 2 — hãy kết hợp "ghi đè" với "trễ một nhịp" của Cách 3. Ví dụ `[23,2,4,6,7]`, `k=6` -> `2`; `[1,5,4,6,7]`, `k=1` -> `2`.

In [42]:
def shortest_good_length(nums, k):
    last = {}                    # số dư -> chỉ số GẦN NHẤT, chỉ gồm các tiền tố đã "đủ xa"
    prev_s, prev_idx = 0, -1     # tiền tố liền trước (chưa đủ xa)
    best = 0
    for i, x in enumerate(nums):
        s = (prev_s + x) % k
        # TODO: nếu s có trong last thì ứng viên độ dài là i - last[s]; cập nhật best (nhỏ nhất, 0 nghĩa là chưa có)
        # TODO: thêm tiền tố liền trước vào last: last[prev_s] = prev_idx
        # TODO: cập nhật prev_s, prev_idx cho vòng sau
        pass
    return best

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_check(nums, k)` theo **Cách 3** (set trễ một nhịp).

In [43]:
def my_check(nums, k):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `longest_good_length(nums, k)`: độ dài **dài nhất** của mảng con tốt (độ dài ≥ 2, tổng chia hết `k`), `0` nếu không có. (b) Viết `count_divisible(nums, k)`: đếm số mảng con **mọi độ dài ≥ 1** có tổng chia hết `k` (giống LeetCode 974 nhưng ở đây ta chỉ xét số không âm). Ví dụ thử `count_divisible([4,5,0,3], 3)`.

In [44]:
def longest_good_length(nums, k):
    # TODO
    pass

def count_divisible(nums, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: với mỗi mảng, tìm hai tiền tố cùng số dư cách xa ít nhất 2. `[0]` chỉ có một phần tử.
- Bài 2: `S_{-1} = 0`, rồi cộng từng phần tử và lấy `% 3`.
- Bài 3: khác bài gốc: ta muốn đoạn **ngắn** nhất nên cần chỉ số **gần nhất** (mới nhất) — nhưng "gần nhất" không được gần hơn 2 chỗ.
- Bài 4: nhớ biến `prev` và thứ tự "kiểm tra rồi mới thêm `prev`".
- Bài 5: dài nhất → chỉ số đầu tiên; đếm → `Counter`.

**Mức 2 (vừa):**
- Bài 2: `4 % 3 = 1`, `(1 + 5) % 3 = 0`, `(0 + 0) % 3 = 0`, `(0 + 3) % 3 = 0`.
- Bài 3: dùng `last` được ghi **trễ một nhịp**: sau khi xử lý `s`, `last[prev_s] = prev_idx`, rồi `prev_s, prev_idx = s, i`. Khi `s in last`, độ dài là `i - last[s]` (luôn >= 2).
- Bài 5(a): giống Cách 2 nhưng không `return`; cập nhật `best = max(best, i - d[s])`. Bài 5(b): `cnt = Counter({0: 1})`; với mỗi tiền tố `ans += cnt[s]; cnt[s] += 1`.

**Mức 3 (gần đáp án):**
- Bài 3: nếu cập nhật `last[s] = i` ngay (không trễ), lần sau ta chỉ thấy chỉ số cách 1 và bỏ lỡ chỉ số cách 2. Vì vậy phải trễ: ghi `prev` sau khi đã tra `s`.
- Bài 4: xem Cách 3 ở mục 7.2.
- Bài 5(b): xem mục 15.3 nhưng không cần "trễ": mọi tiền tố cũ đều hợp lệ (độ dài ≥ 1).

## 19. Đáp án tham khảo

In [45]:
# Bài 1: dự đoán True, False, False, False
assert Solution().checkSubarraySum([1, 2, 3], 5) is True        # 2 + 3
assert Solution().checkSubarraySum([1, 2, 3], 7) is False
assert Solution().checkSubarraySum([0], 1) is False             # độ dài 1
assert Solution().checkSubarraySum([6, 1], 6) is False          # 6+1 = 7
print("Bài 1 OK")

# Bài 2
ans2_ref = [0, 1, 0, 0, 0]
cur, acc = 0, [0]
for x in [4, 5, 0, 3]:
    cur = (cur + x) % 3
    acc.append(cur)
assert acc == ans2_ref
print("Bài 2:", ans2_ref, "-> các chỉ số số dư 0 là -1, 1, 2, 3 -> có nhiều đoạn tốt")

Bài 1 OK
Bài 2: [0, 1, 0, 0, 0] -> các chỉ số số dư 0 là -1, 1, 2, 3 -> có nhiều đoạn tốt


In [46]:
# Bài 3
def shortest_good_length_ref(nums, k):
    last = {}
    prev_s, prev_idx = 0, -1
    best = 0
    for i, x in enumerate(nums):
        s = (prev_s + x) % k
        if s in last:
            length = i - last[s]
            best = length if best == 0 else min(best, length)
        last[prev_s] = prev_idx        # trễ một nhịp: chỉ ghi sau khi đã tra
        prev_s, prev_idx = s, i
    return best

def shortest_oracle(nums, k):
    best = 0
    for i, j in combinations(range(len(nums)), 2):
        if sum(nums[i:j + 1]) % k == 0:
            length = j - i + 1
            best = length if best == 0 else min(best, length)
    return best

assert shortest_good_length_ref([23, 2, 4, 6, 7], 6) == 2
assert shortest_good_length_ref([1, 5, 4, 6, 7], 1) == 2
assert shortest_good_length_ref([5], 5) == 0
rng = random.Random(1903)
for _ in range(1500):
    nums = [rng.randint(0, 8) for _ in range(rng.randint(1, 10))]
    k = rng.randint(1, 9)
    assert shortest_good_length_ref(nums, k) == shortest_oracle(nums, k), (nums, k)
print("Bài 3 OK")

Bài 3 OK


In [47]:
# Bài 4
def my_check_ref(nums, k):
    seen, prev, s = set(), 0, 0
    for x in nums:
        s = (s + x) % k
        if s in seen:
            return True
        seen.add(prev)
        prev = s
    return False

rng = random.Random(1904)
for _ in range(1500):
    nums = [rng.randint(0, 8) for _ in range(rng.randint(1, 10))]
    k = rng.randint(1, 9)
    assert my_check_ref(nums, k) == oracle(nums, k)
print("Bài 4 OK")

Bài 4 OK


In [48]:
# Bài 5
def longest_good_length_ref(nums, k):
    d = {0: -1}
    s = 0
    best = 0
    for i, x in enumerate(nums):
        s = (s + x) % k
        if s not in d:
            d[s] = i
        elif i - d[s] >= 2:
            best = max(best, i - d[s])
    return best

def count_divisible_ref(nums, k):
    cnt = Counter({0: 1})
    s = ans = 0
    for x in nums:
        s = (s + x) % k
        ans += cnt[s]
        cnt[s] += 1
    return ans

def longest_oracle(nums, k):
    best = 0
    for i, j in combinations(range(len(nums)), 2):
        if sum(nums[i:j + 1]) % k == 0:
            best = max(best, j - i + 1)
    return best

def count_div_oracle(nums, k):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n) if sum(nums[i:j + 1]) % k == 0)

assert longest_good_length_ref([23, 2, 6, 4, 7], 6) == 5
assert count_divisible_ref([4, 5, 0, 3], 3) == count_div_oracle([4, 5, 0, 3], 3)
rng = random.Random(1905)
for _ in range(1500):
    nums = [rng.randint(0, 8) for _ in range(rng.randint(1, 10))]
    k = rng.randint(1, 9)
    assert longest_good_length_ref(nums, k) == longest_oracle(nums, k), (nums, k)
    assert count_divisible_ref(nums, k) == count_div_oracle(nums, k), (nums, k)
print("Bài 5 OK; count_divisible([4,5,0,3], 3) =", count_divisible_ref([4, 5, 0, 3], 3))

Bài 5 OK; count_divisible([4,5,0,3], 3) = 6


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template prefix sum theo modulo (chỉ số đầu tiên):**

```python
class Solution:
    def checkSubarraySum(self, nums, k):
        d = {0: -1}                 # tiền tố rỗng: số dư 0 ở chỉ số -1
        s = 0
        for i, x in enumerate(nums):
            s = (s + x) % k
            if s not in d:
                d[s] = i            # chỉ ghi lần đầu
            elif i - d[s] > 1:      # độ dài >= 2
                return True
        return False
```

**Template set trễ một nhịp:**

```python
seen, prev, s = set(), 0, 0
for x in nums:
    s = (s + x) % k
    if s in seen:
        return True
    seen.add(prev)                  # thêm sau khi kiểm tra
    prev = s
return False
```

**Câu thần chú:** *cùng số dư tiền tố ⇒ đoạn giữa chia hết; nhớ chỉ số đầu tiên; cách nhau ≥ 2; đừng quên mồi `{0: -1}`.*

**Checklist trước khi submit:**

- [ ] Đã mồi `d = {0: -1}`?
- [ ] Chỉ ghi `d[s] = i` khi `s` chưa có trong `d` (không ghi đè)?
- [ ] Điều kiện độ dài `i - d[s] > 1` (≥ 2), không phải `>= 1`?
- [ ] Đã lấy `% k` mỗi bước (hoặc ít nhất khi tra bảng)?
- [ ] Đã thử: `[x]` một phần tử, `[0,0]`, `k = 1`, `k` lớn hơn tổng, mảng cả đoạn từ đầu?
- [ ] Nhớ `0` là bội của `k` và `k >= 1` nên không chia cho 0?